# Pediatric Genetic Disorder — Final Submission

**Outputs:**
```
Patient Id | Genetic Disorder | Disorder Subclass
```


In [1]:
!pip install -q lightgbm catboost xgboost

import numpy as np
import pandas as pd
import pickle, os, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

np.random.seed(42)
print('✅ Libraries loaded')

✅ Libraries loaded


In [5]:

PREP_PATH = '/kaggle/input/datasets/masteranany/mydata/'


RAW_PATH  = '/kaggle/input/datasets/masteranany/mydata/'

OUT_PATH  = '/kaggle/working/'

In [7]:
# ── Load preprocessed features & labels ────────────────────────────────────────
X_train = pd.read_csv(PREP_PATH + 'X_train_preprocessed.csv')
X_test  = pd.read_csv(PREP_PATH + 'X_test_preprocessed.csv')
y1_train = np.load(PREP_PATH + 'y1_train.npy')   # Genetic Disorder
y2_train = np.load(PREP_PATH + 'y2_train.npy')   # Disorder Subclass

with open(PREP_PATH + 'le_t1.pkl', 'rb') as f:
    le_t1 = pickle.load(f)          # Genetic Disorder encoder
with open(PREP_PATH + 'le_t2.pkl', 'rb') as f:
    le_t2 = pickle.load(f)          # Disorder Subclass encoder

raw_test = pd.read_csv('/kaggle/input/datasets/masteranany/newgec/test.csv')
patient_ids = raw_test['Patient Id'].values

print(f'X_train  : {X_train.shape}  |  X_test  : {X_test.shape}')
print(f'y1 classes: {np.unique(y1_train)}  |  y2 classes: {np.unique(y2_train)}')
print(f'Patient Ids: {len(patient_ids)} → first 3: {patient_ids[:3]}')
print(f'le_t1 classes: {list(le_t1.classes_)}')
print(f'le_t2 classes: {list(le_t2.classes_)}')

X_train  : (39474, 53)  |  X_test  : (9465, 53)
y1 classes: [0 1 2 3]  |  y2 classes: [0 1 2 3 4 5 6 7 8 9]
Patient Ids: 9465 → first 3: ['PID0x4175' 'PID0x21f5' 'PID0x49b8']
le_t1 classes: ['Mitochondrial genetic inheritance disorders', 'Multifactorial genetic inheritance disorders', 'Single-gene inheritance diseases', nan]
le_t2 classes: ["Alzheimer's", 'Cancer', 'Cystic fibrosis', 'Diabetes', 'Hemochromatosis', "Leber's hereditary optic neuropathy", 'Leigh syndrome', 'Mitochondrial myopathy', 'Tay-Sachs', nan]


In [9]:
# ── Helper: build fast model set ────────────────────────────────────────────────
def make_models(n_classes):
    """Returns (xgb, lgb, cat) tuned for n_classes."""
    # ── XGBoost ──────────────────────────────────────────────────────────────
    xgb = XGBClassifier(
        n_estimators      = 500,
        learning_rate     = 0.08,
        max_depth         = 7,
        subsample         = 0.80,
        colsample_bytree  = 0.80,
        min_child_weight  = 3,
        reg_alpha         = 0.1,
        reg_lambda        = 1.5,
        eval_metric       = 'mlogloss',
        **(dict(objective='multi:softprob', num_class=n_classes)
           if n_classes > 2 else {}),
        random_state=42, n_jobs=-1
    )
    # ── LightGBM ─────────────────────────────────────────────────────────────
    lgb = LGBMClassifier(
        n_estimators      = 600,
        learning_rate     = 0.08,
        num_leaves        = 63,
        max_depth         = 8,
        subsample         = 0.80,
        colsample_bytree  = 0.80,
        min_child_samples = 20,
        reg_alpha         = 0.1,
        reg_lambda        = 1.5,
        **(dict(objective='multiclass', num_class=n_classes)
           if n_classes > 2 else {}),
        random_state=42, n_jobs=-1, verbose=-1
    )
    # ── CatBoost ─────────────────────────────────────────────────────────────
    cat = CatBoostClassifier(
        iterations        = 500,
        learning_rate     = 0.08,
        depth             = 8,
        l2_leaf_reg       = 3.0,
        bagging_temperature = 0.5,
        border_count      = 128,
        **(dict(loss_function='MultiClass', classes_count=n_classes)
           if n_classes > 2 else {}),
        random_seed=42, verbose=0
    )
    return xgb, lgb, cat


N1 = len(np.unique(y1_train))   # 4 classes
N2 = len(np.unique(y2_train))   # 10 classes

xgb1, lgb1, cat1 = make_models(N1)
xgb2, lgb2, cat2 = make_models(N2)

print(f'Target 1 → {N1} classes | Target 2 → {N2} classes')
print('Models ready ✅')

Target 1 → 4 classes | Target 2 → 10 classes
Models ready ✅


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# TARGET 1 — Genetic Disorder (4 classes)
# ══════════════════════════════════════════════════════════════════════════════
import time

print('=== TARGET 1: Genetic Disorder ===')

# Soft-voting ensemble  (faster than stacking, nearly same accuracy)
ensemble1 = VotingClassifier(
    estimators=[('xgb', xgb1), ('lgb', lgb1), ('cat', cat1)],
    voting='soft', n_jobs=-1
)

# Quick 3-fold CV for a sanity score
CV3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
t0 = time.time()
cv_scores_1 = cross_val_score(ensemble1, X_train, y1_train,
                               cv=CV3, scoring='f1_macro', n_jobs=1)
print(f'3-fold CV F1-macro: {cv_scores_1.mean():.4f} ± {cv_scores_1.std():.4f}  '
      f'({time.time()-t0:.0f}s)')

# Train on FULL data
print('Training on full data ...')
ensemble1.fit(X_train, y1_train)

# Predict (integer classes)
y1_pred_int = ensemble1.predict(X_test)

# Decode → original string labels
y1_pred_str = le_t1.inverse_transform(y1_pred_int)



=== TARGET 1: Genetic Disorder ===
3-fold CV F1-macro: 0.6925 ± 0.0034  (160s)
Training on full data ...


TypeError: '<' not supported between instances of 'float' and 'str'

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# TARGET 2 — Disorder Subclass (10 classes)
# ══════════════════════════════════════════════════════════════════════════════
print('\n=== TARGET 2: Disorder Subclass ===')

ensemble2 = VotingClassifier(
    estimators=[('xgb', xgb2), ('lgb', lgb2), ('cat', cat2)],
    voting='soft', n_jobs=-1
)

t0 = time.time()
cv_scores_2 = cross_val_score(ensemble2, X_train, y2_train,
                               cv=CV3, scoring='f1_macro', n_jobs=1)
print(f'3-fold CV F1-macro: {cv_scores_2.mean():.4f} ± {cv_scores_2.std():.4f}  '
      f'({time.time()-t0:.0f}s)')

# Train on FULL data
print('Training on full data ...')
ensemble2.fit(X_train, y2_train)

# Predict (integer classes)
y2_pred_int = ensemble2.predict(X_test)

# Decode → original string labels
y2_pred_str = le_t2.inverse_transform(y2_pred_int)

print(' T2 done. ' )


=== TARGET 2: Disorder Subclass ===
3-fold CV F1-macro: 0.5478 ± 0.0044  (390s)
Training on full data ...
 T2 done. 


In [13]:
result = pd.DataFrame({
    'Patient Id':        patient_ids,
    'Genetic Disorder':  y1_pred_str,
    'Disorder Subclass': y2_pred_str
})

print('=== PREVIEW (first 10 rows) ===')
print(result.head(10).to_string(index=False))
print(f'\nTotal rows: {len(result)}')

=== PREVIEW (first 10 rows) ===
Patient Id                             Genetic Disorder                   Disorder Subclass
 PID0x4175  Mitochondrial genetic inheritance disorders Leber's hereditary optic neuropathy
 PID0x21f5  Mitochondrial genetic inheritance disorders              Mitochondrial myopathy
 PID0x49b8  Mitochondrial genetic inheritance disorders                           Tay-Sachs
 PID0x2d97  Mitochondrial genetic inheritance disorders                      Leigh syndrome
 PID0x58da             Single-gene inheritance diseases                     Cystic fibrosis
 PID0x96b6 Multifactorial genetic inheritance disorders                            Diabetes
  PID0x399  Mitochondrial genetic inheritance disorders                     Cystic fibrosis
 PID0x6819 Multifactorial genetic inheritance disorders                            Diabetes
 PID0x9697  Mitochondrial genetic inheritance disorders              Mitochondrial myopathy
 PID0x628a  Mitochondrial genetic inheritance di

In [15]:
print(f'Shape          : {result.shape}')
print(f'Null Patient Id: {result["Patient Id"].isnull().sum()}')
print(f'Null T1        : {result["Genetic Disorder"].isnull().sum()}')
print(f'Null T2        : {result["Disorder Subclass"].isnull().sum()}')


print('\n=== T1 DISTRIBUTION ===')
print(result['Genetic Disorder'].value_counts().to_string())

print('\n=== T2 DISTRIBUTION ===')
print(result['Disorder Subclass'].value_counts().to_string())

Shape          : (9465, 3)
Null Patient Id: 0
Null T1        : 29
Null T2        : 44

=== T1 DISTRIBUTION ===
Genetic Disorder
Mitochondrial genetic inheritance disorders     5479
Single-gene inheritance diseases                2182
Multifactorial genetic inheritance disorders    1775

=== T2 DISTRIBUTION ===
Disorder Subclass
Leigh syndrome                         2902
Cystic fibrosis                        2019
Diabetes                               1920
Mitochondrial myopathy                 1515
Tay-Sachs                               720
Hemochromatosis                         157
Leber's hereditary optic neuropathy     141
Alzheimer's                              45
Cancer                                    2


In [17]:
os.makedirs(OUT_PATH, exist_ok=True)

out_file = OUT_PATH + 'result.csv'
result.to_csv(out_file, index=False)

# Verify the saved file matches the expected format
check = pd.read_csv(out_file)
print(f' result.csv saved → {out_file}')
print(f'   Rows: {len(check)} | Columns: {list(check.columns)}')
print(f'\nFirst 5 rows of saved file:')
print(check.head(5).to_string(index=False))

print(f' T1 CV F1-macro : {cv_scores_1.mean():.4f}')
print(f' T2 CV F1-macro : {cv_scores_2.mean():.4f}')


 result.csv saved → /kaggle/working/result.csv
   Rows: 9465 | Columns: ['Patient Id', 'Genetic Disorder', 'Disorder Subclass']

First 5 rows of saved file:
Patient Id                            Genetic Disorder                   Disorder Subclass
 PID0x4175 Mitochondrial genetic inheritance disorders Leber's hereditary optic neuropathy
 PID0x21f5 Mitochondrial genetic inheritance disorders              Mitochondrial myopathy
 PID0x49b8 Mitochondrial genetic inheritance disorders                           Tay-Sachs
 PID0x2d97 Mitochondrial genetic inheritance disorders                      Leigh syndrome
 PID0x58da            Single-gene inheritance diseases                     Cystic fibrosis
 T1 CV F1-macro : 0.6925
 T2 CV F1-macro : 0.5478
 Output file    : /kaggle/working/result.csv
